In [ ]:
import json
import httpx
import openai
from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI()
BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


def get_popular_movies():
    response = httpx.get(f"{BASE_URL}/movies")
    return response.json()


def get_movie_details(id):
    response = httpx.get(f"{BASE_URL}/movies/{id}")
    return response.json()


def get_movie_credits(id):
    response = httpx.get(f"{BASE_URL}/movies/{id}/credits")
    return response.json()


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 특정 영화의 상세 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 ID"}},
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 ID로 출연진 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer", "description": "영화 ID"}},
                "required": ["id"],
            },
        },
    },
]

function_map = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}


def run_agent(user_message):
    print(f"\n사용자: {user_message}")
    messages = [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        tools=tools,
        messages=messages,
    )

    message = response.choices[0].message

    if message.tool_calls:
        messages.append(message)

        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            print(f"호출 함수: {func_name}({args})")

            result = function_map[func_name](**args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        print(f"응답: {final_response.choices[0].message.content}")
    else:
        print(f"응답: {message.content}")


# 테스트
run_agent("지금 인기 있는 영화가 무엇인지 알려줘")
run_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")
run_agent("movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘")